# Rich Chess Ebook — extraction pipeline

Turns a **PDF chess book** into a `.rce` archive the Flutter app can read.

Two notations are supported: figurine Unicode, and plain letters in `en`, `fr`, `de`,
`es`, `it` or `nl`. Both are read from the text layer.

A book whose piece symbols are **drawn** — a scan, or a figurine font — has no readable
symbols in that layer at all, and needs step 4b: a trained classifier reads them off the
page images and writes them back in as figurines. Step 4 says whether this book is one
of those.

The logic is not in these cells but in the repository's `rce_pipeline` package: a
notebook is neither testable nor readable as a diff, whereas each step here is a module
that can be fixed and re-run on its own.

| Step | Module | Artefact written |
| --- | --- | --- |
| 1. Text + per-character geometry | `extract.py` | `work/01_pages.json` |
| 1c. Piece symbols read off the image | `scan.py`, `glyphs.py` | `work/01b_glyphs.json` |
| 2. Notation detection | `notation.py` | `work/02_notation.json` |
| 3a. Tokenising | `tokenize.py` | `work/03_tokens.json` |
| 3b + 4. Move tree, legality, FEN | `parse.py` | `work/04_moves.json` |
| 5. Packaging | `package.py` | `book.rce` |

Each step reads the previous one's artefact, so editing `parse.py` and re-running does
not redo extraction — by far the slowest part.

## 1. Install

In [ ]:
!pip install -q pymupdf chess
# Only needed for a scanned or figurine-font book (step 4b):
!pip install -q scikit-learn scikit-image pillow

## 2. Get the pipeline code

Set `REPO_URL` if the repository is on GitHub. Otherwise leave it at `None` and the
cell will ask you to upload a ZIP of the `pipeline/` directory.

In [ ]:
REPO_URL = None  # e.g. "https://github.com/<user>/RichChessEbooks.git"

import os, sys, zipfile

if REPO_URL:
    if not os.path.isdir("RichChessEbooks"):
        !git clone --depth 1 $REPO_URL
    PIPELINE_DIR = "RichChessEbooks/pipeline"
elif os.path.isdir("pipeline"):
    PIPELINE_DIR = "pipeline"
else:
    from google.colab import files
    print("Upload a ZIP containing the pipeline/ directory")
    for name in files.upload():
        with zipfile.ZipFile(name) as archive:
            archive.extractall(".")
    PIPELINE_DIR = "pipeline"

sys.path.insert(0, os.path.abspath(PIPELINE_DIR))

import rce_pipeline
from rce_pipeline import extract, notation, tokenize, parse, package, pipeline
print("rce_pipeline", rce_pipeline.__version__, "from", PIPELINE_DIR)

## 3. Load the PDF

Either upload it directly, or read it from Google Drive if the file is large.

In [ ]:
PDF_PATH = None  # e.g. "/content/drive/MyDrive/books/my_book.pdf"

if PDF_PATH is None:
    from google.colab import files
    uploaded = files.upload()
    PDF_PATH = next(iter(uploaded))

print(PDF_PATH, "—", extract.page_count(PDF_PATH), "pages")

## 4. Step 2 alone — which notation?

Read a sample first, to find out what you are dealing with. The result is a
**conclusion to confirm**, not a question asked cold: the counts supporting it are
printed underneath.

Two things to check here.

**Style.** `figurine_unicode` and `letters` are read straight from the text layer.
`figurine_font` is not: the layer holds latin letters bearing no relation to the pieces
drawn, and the symbols have to be recovered from the page images in step 4b.

**Whether this is a scan.** If the raw text below shows moves as garbage — `4)xf7`,
`We6`, `2xf7` — the layer is OCR output. Its prose is usually fine and its move numbers
and squares are mostly right; only the piece symbols are hopeless, which is again what
step 4b is for. Prose that reads cleanly while moves do not is the signature.

`report.needs_glyph_recovery` answers the question in one boolean, and covers a third
case the two above miss: a book full of pawn moves that names no piece in any language.
That only happens when the symbols are drawn.

In [ ]:
SAMPLE_FIRST_PAGE = 1
SAMPLE_LAST_PAGE = 40

sample = extract.extract_pages(PDF_PATH, first_page=SAMPLE_FIRST_PAGE, last_page=SAMPLE_LAST_PAGE)
report = notation.detect_notation(sample)
print(report.summary())
print()
print("Most frequent fonts:")
for name, count in list(extract.font_inventory(sample).items())[:8]:
    print(f"  {count:>7}  {name}")

In [ ]:
# Raw text sample, to see with your own eyes what the text layer holds.
page = sample[len(sample) // 2]
print(f"--- page {page.number} ({page.width} x {page.height} pt) ---")
print(page.text[:1200])

## 4b. Books whose piece symbols are only in the image

Skip this if step 4 reported `needs_glyph_recovery = False`.

The classifier is a random forest over five classes — King, Queen, Rook, Bishop, Knight
— trained outside this repository; upload `chess_glyphs_classifier.zip` below. What it
does not have is a "not a piece" class, so a crop is only shown to it when its shape
already says piece: about twice as wide as the page's letters, and nearly square. What
comes back above `min_confidence` is written into the pages as a figurine, carrying the
box of the printed symbol rather than of whatever the scanner read there.

Measured on two hand-read pages of a French scanned book, 54 symbols printed: **53
recovered, none invented**, at the default confidence of 0.45. Lower it and phantom
pieces appear in the prose; raise it and bishops start dropping out first.

In [ ]:
GLYPH_MODEL = None  # e.g. "/content/drive/MyDrive/chess_glyphs_classifier.zip"

if GLYPH_MODEL is None:
    from google.colab import files
    print("Upload chess_glyphs_classifier.zip")
    GLYPH_MODEL = next(iter(files.upload()))

from rce_pipeline import glyphs

classifier = glyphs.GlyphClassifier.load(GLYPH_MODEL)
print("loaded", GLYPH_MODEL)

### What it finds, before committing to it

Two pages, the symbols found on them, and the text they will be written into. Read the
`after` lines against the page itself: this is the check that matters, and no counter
replaces it.

In [ ]:
PREVIEW_PAGES = (SAMPLE_FIRST_PAGE, SAMPLE_FIRST_PAGE + 1)
MIN_CONFIDENCE = glyphs.DEFAULT_MIN_CONFIDENCE

from rce_pipeline import scan

preview = extract.extract_pages(PDF_PATH, first_page=PREVIEW_PAGES[0], last_page=PREVIEW_PAGES[1])
before = {p.number: [l.text for l in scan.notation_lines(scan.segment_lines(p))] for p in preview}
repaired, found = glyphs.recover_pieces(
    PDF_PATH, preview, classifier, min_confidence=MIN_CONFIDENCE
)

placed, total = glyphs.placement_score(repaired)
if total:
    print(f"{total} symbols recovered, {placed} spliced into a move ({placed / total:.0%})\n")
else:
    print("no symbols found — check MIN_CONFIDENCE, and that these pages carry moves\n")
for page in repaired:
    for old, new in zip(before[page.number], [l.text for l in scan.notation_lines(scan.segment_lines(page))]):
        if old != new:
            print(f"  before: {old}\n  after : {new}\n")

A low "spliced into a move" share means the symbols were recognised and then written
into the wrong characters. That is not the classifier: it is the text layer's boxes.
Tesseract divides a word's box evenly among the characters it read, so a layer that read
`tZJg3` where `♘g3` is printed puts every box in the wrong place, and the symbol lands
beside the move instead of at its head. Around 90% is a well-boxed layer; below 60% the
moves from this book are not worth parsing, and the page images would have to be read in
full rather than repaired.

In [ ]:
# The crops the classifier was shown, with what it made of them.
from PIL import Image
import io

page = repaired[0]
source = next(p for p in preview if p.number == page.number)
lines = scan.notation_lines(scan.segment_lines(source))
with scan.PageRenderer(PDF_PATH) as renderer:
    for line in lines[:4]:
        image = renderer.crop(line)
        display(Image.open(io.BytesIO(image.png)))
        on_line = [g for g in found if g.page == page.number
                   and line.bbox.y <= g.bbox.y + g.bbox.h / 2 <= line.bbox.y + line.bbox.h]
        print("  ".join(f"{g.figurine} {g.confidence:.2f}" for g in sorted(on_line, key=lambda g: g.bbox.x)) or "(nothing)")

## 5. Full pipeline

`FIRST_PAGE` / `LAST_PAGE` restrict the work to part of the book — start small, on a
chapter whose content you know, before launching 400 pages.

- `strict_numbering=True` only reads a move when a move number has just announced one,
  or when variation brackets make the context unambiguous. That is what separates `Bb5`
  from a figure caption reading "diagram b4". Set it to `False` for a book that prints
  long unnumbered sequences.
- `force_notation` bypasses step 2, useful when the sample was too short to reach the
  detection threshold.
- `glyph_model` turns on step 4b, and is only used when the book needs it. It costs a
  rendering pass over every line carrying a move number.
- **`force_language` is worth setting whenever you know the book.** It decides which
  alphabet piece initials come from, and the alphabets overlap: `R` is the King in
  French and the Rook in English. Both readings are frequently legal in the same
  position, so a wrong language does not fail — it produces a different game.

| Language | King | Queen | Rook | Bishop | Knight |
| --- | --- | --- | --- | --- | --- |
| `en` | K | Q | R | B | N |
| `fr` | R | D | T | F | C |
| `de` | K | D | T | L | S |
| `es` / `it` | R | D | T | A | C |
| `nl` | K | D | T | L | P |

In [ ]:
FIRST_PAGE = 1
LAST_PAGE = 40
OUTPUT = "/content/book.rce"

result = pipeline.run(
    PDF_PATH,
    work_dir="/content/work",
    output_path=OUTPUT,
    first_page=FIRST_PAGE,
    last_page=LAST_PAGE,
    strict_numbering=True,
    force_notation=None,  # or "figurine_unicode" / "letters"
    force_language=None,  # or "fr" / "en" / "de" / "es" / "it" / "nl"
    glyph_model=None,     # or GLYPH_MODEL, for a scanned or figurine-font book
)
print(result.report())

## 6. What did not get through

`broken` first: no legal reading was found, so the move is a hole in the line. Then
`uncertain`, accepted after repairing a look-alike scanning error (`0`/`O`, `1`/`l`,
`8`/`B`) — the `repair` field says what was substituted.

Repairs are deliberately conservative. Allowing one arbitrary wrong character would
recover more moves and would also turn `Qh9` into `Qh5` and `Nc6` into `Nc3`: squares
differ by a single character all the time, so the pipeline would emit legal but wrong
moves that silently corrupt every position further down the line.

These moves keep their page and their box, so they stay clickable in the app — which is
where they are meant to be corrected.

In [ ]:
for move in result.problems(limit=25):
    detail = move.repair["reason"] if move.repair else ""
    print(f"[{move.status:>9}] p.{move.page:>3}  {move.san:<8} conf={move.confidence:.2f}  {detail}")

print(f"\n{len(result.parsed.skipped)} tokens dropped before validation:")
for skipped in result.parsed.skipped[:15]:
    print(f"  p.{skipped['page']:>3}  {skipped['text']:<10} {skipped['reason']}")

## 7. Check the boxes by eye

This is the most useful check in the notebook. A box off by a few points shows up in no
counter, but makes the clickable zone useless in the app. So render the page and draw
the boxes on top of it.

The code converts `.rce` coordinates (origin bottom-left) back to MuPDF's (origin
top-left) — the same round trip Flutter makes, in reverse. If the frames land on the
moves, the convention is right on both sides.

In [ ]:
try:
    import pymupdf as fitz
except ImportError:
    import fitz
from PIL import Image, ImageDraw

PREVIEW_PAGE = result.parsed.moves[0].page if result.parsed.moves else FIRST_PAGE
ZOOM = 2.0
STATUS_COLOURS = {"ok": (0, 160, 0), "uncertain": (220, 140, 0), "broken": (210, 0, 0)}

doc = fitz.open(PDF_PATH)
page = doc[PREVIEW_PAGE - 1]
pixmap = page.get_pixmap(matrix=fitz.Matrix(ZOOM, ZOOM))
image = Image.frombytes("RGB", (pixmap.width, pixmap.height), pixmap.samples)
draw = ImageDraw.Draw(image)

page_height = page.rect.height
drawn = 0
for move in result.parsed.moves:
    if move.page != PREVIEW_PAGE:
        continue
    b = move.bbox
    top = page_height - b.y - b.h  # flip back to MuPDF's top-left origin
    draw.rectangle(
        [b.x * ZOOM, top * ZOOM, (b.x + b.w) * ZOOM, (top + b.h) * ZOOM],
        outline=STATUS_COLOURS[move.status],
        width=2,
    )
    drawn += 1

doc.close()
print(f"{drawn} boxes drawn on page {PREVIEW_PAGE}")
image

## 8. Check the move tree

Variations are reconstructed from `parent_id`, never from array order. This display
follows those links, which checks along the way that they are coherent.

In [ ]:
from collections import defaultdict

GAME_INDEX = 0     # which game to show
MAX_LINES = 80

game = result.parsed.games[GAME_INDEX]
children = defaultdict(list)
for move in result.parsed.moves:
    if move.game_id == game.id:
        children[move.parent_id].append(move)
for siblings in children.values():
    siblings.sort(key=lambda m: m.variation_index)

printed = 0

def show(move_id, depth):
    global printed
    for child in children[move_id]:
        if printed >= MAX_LINES:
            return
        printed += 1
        number = f"{(child.ply + 1) // 2}." + ("" if child.ply % 2 else "..")
        mark = {"ok": " ", "uncertain": "~", "broken": "!"}[child.status]
        note = f"    [{child.comment[:60]}]" if child.comment else ""
        print("  " * depth + f"{mark} {number}{child.san}{note}")
        # Only a variation shifts the indentation; the main line stays flush.
        show(child.id, depth + 1 if child.variation_index else depth)

title = game.title or "(untitled)"
print(f"=== {game.id} - {title} (p.{game.page_start}) ===")
show(None, 0)

## 9. Download the archive

The `.rce` holds the original PDF **unchanged**, plus `manifest.json` and `moves.json`.
This is the file the Flutter app imports.

In [ ]:
import zipfile

with zipfile.ZipFile(OUTPUT) as archive:
    for info in archive.infolist():
        print(f"{info.file_size:>12,} B  {info.filename}")

from google.colab import files
files.download(OUTPUT)